In [1]:
import pandas as pd
import numpy as np
import re

In [2]:

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
notes2025 = PROJECT_ROOT / "Data" / "RawTXT" / "2025 Reviews.txt"



    

In [3]:


# Read the file
# utf-8-sig automatically removes a UTF-8 BOM if one exists
with open(notes2025, "r", encoding="utf-8-sig") as f:
    text = f.read()

header_pattern = re.compile(
    r'(?m)^\s*-?\d+\s*:\s*?\s+(.+?)\s+by\s+(.+?)\s*$'
)

headers = list(header_pattern.finditer(text))

print("Headers found:", len(headers))

rows = []

for i, header in enumerate(headers):

    # Get everything from this header to the next header
    start = header.start()
    end = headers[i + 1].start() if i + 1 < len(headers) else len(text)

    review = text[start:end]

    # Correct capture groups:
    # group(1) = album name
    # group(2) = artist name
    album_name = header.group(1).strip()
    artist_name = header.group(2).strip()

    # Everything after the header
    notes_and_score = review[header.end() - start:].strip()

    # Score is the final number/number
    score_match = re.search(
        r'(\d+(?:\.\d+)?/\d+)\s*$',
        notes_and_score
    )

    if score_match:
        score = score_match.group(1)
        notes = notes_and_score[:score_match.start()].strip()
    else:
        score = None
        notes = notes_and_score

    rows.append({
        "album name": album_name,
        "artist name": artist_name,
        "score": score,
        "notes": notes
    })

df = pd.DataFrame(rows)

print(df.shape)
print(df.head())




Headers found: 224
(224, 4)
            album name                         artist name   score  \
0           KPOP Bitch                            The Deep    7/10   
1      The Provocateur                               Adela  6.5/10   
2              Holoboy                     This Is Lorelei  7.5/10   
3               Shiraz                       Dhafer Yousef    9/10   
4  Dancing on The Edge  Ryan Davis and the  Roadhouse Band    8/10   

                                               notes  
0  Hyperpop kpop. It's fun. Lucky star is great. ...  
1  First song of hyoerpop album us a straight ban...  
2  The latest solo record by 1/2 of Water from yo...  
3  Middle eastern jazz from 2025. Nice trumpet wo...  
4  Their 2023 debut. Still witty and charming wit...  


In [23]:
print(text.splitlines()[0])
print([ord(c) for c in text.splitlines()[0]])

226: KPOP Bitch by The Deep
[50, 50, 54, 58, 32, 75, 80, 79, 80, 32, 66, 105, 116, 99, 104, 32, 98, 121, 32, 84, 104, 101, 32, 68, 101, 101, 112]


In [24]:
headers = list(header_pattern.finditer(text))
# Extract the album numbers that were successfully found
found_numbers = []

for header in headers:
    # Get the text immediately before the album title
    line = header.group(0)
    number_match = re.match(r'\s*(\d+)/(\d+)', line)

    if number_match:
        found_numbers.append(int(number_match.group(1)))

# Find which album numbers are missing
expected_numbers = set(range(1, 261))
missing_numbers = sorted(expected_numbers - set(found_numbers))

print("Headers found:", len(headers))
print("Missing album numbers:", missing_numbers)


Headers found: 224
Missing album numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 

In [25]:
print(df.shape)

print(df.tail(3))

(224, 4)
                              album name  artist name score  \
221  Time Will Die and Love Will Bury It  Rolo Tomasi  8/10   
222                               Najeem       Nabeel  7/10   
223                         WHAT WE DREW        Yaeji  7/10   

                                                 notes  
221  Opens with patchy synth ambience and slight ch...  
222  Iraqi DIY shoegaze. It's great and very MBV in...  
223  An sort of funky beat into very processed voca...  


In [26]:
review_numbers = []

for header in headers:
    match = re.match(r'\s*-?(\d+)', header.group(0))
    if match:
        review_numbers.append(int(match.group(1)))

print("Reviews parsed:", len(review_numbers))

# Show the range of review numbers
print("Highest review number:", max(review_numbers))
print("Lowest review number:", min(review_numbers))

# Find gaps
expected = set(range(min(review_numbers), max(review_numbers) + 1))
found = set(review_numbers)

missing = sorted(expected - found)

print("Missing review numbers:", missing)


Reviews parsed: 224
Highest review number: 226
Lowest review number: 1
Missing review numbers: [23, 43]


In [27]:
df.iloc[168]

album name                                         Flying Beagle
artist name                                       Himiko Kikuchi
score                                                       8/10
notes          1987 J Jazz. Starts with a crazy slap bass par...
Name: 168, dtype: str

In [28]:
df.rename(columns={"album name": "album_name", "artist name": "artist_name"}, inplace=True)
#------
# Make sure everything is treated as a string
df['score'] = df['score'].astype(str)

# Convert scores out of 3
df.loc[df['score'].str.endswith('/3'), 'score'] = (
    df.loc[df['score'].str.endswith('/3'), 'score']
    .str.replace('/3', '', regex=False)
    .astype(float) * 2
).astype(str)

# Remove /10
df['score'] = df['score'].str.replace('/10', '', regex=False)

# Convert the entire column to numbers
df['score'] = pd.to_numeric(df['score'])
#-----------------
df.to_csv(
    r"C:\Users\alexw\Projects\ALBUM_PROJ\2025_reviews.csv",
    index=False,
    encoding="utf-8"
)